# Lab 08: Mini-Batch Training, Evaluation, and Convolutional Neural Networks

Welcome to Lab 08! In Lab 07 we built a working neural network using **full-batch gradient descent** on a small toy dataset — we passed the entire dataset through the model at once, computed the gradient, and took one step per epoch. That was a great way to see all the moving parts clearly.

In this lab we **scale up to real image data**. We will cover three main ideas:

1. **Mini-batch training with DataLoader** — the standard approach when your dataset doesn't fit in memory all at once (which is almost always in practice).
2. **A proper evaluation procedure** — a `validate()` function that measures accuracy, precision, recall, and F1 on held-out data.
3. **Convolutional layers** — the building block of image models. You will learn what every parameter does and visualise what these layers actually learn.

By the end of this lab you will have:
- Built and trained a **Convolutional Neural Network (CNN)** on the FashionMNIST dataset
- Implemented the `training()` and `validate()` functions you will need for PS6
- Visualised feature maps to build intuition for what CNNs detect

Let's get started!

---
## Section 0: Setup & Imports

Let's begin by importing everything we'll need for this lab. A few new faces here:

- **`torchvision`** — a companion library to PyTorch that provides popular image datasets, model architectures, and image transforms.
- **`torchvision.transforms`** — tools for preprocessing images (resizing, normalising, converting to tensors, etc.).
- **`torch.utils.data.DataLoader`** — a utility that handles batching, shuffling, and parallel data loading automatically.

If any of these are not installed, you can install them with:
```
pip install torch torchvision scikit-learn matplotlib
```

We also fix random seeds (both PyTorch and NumPy) for reproducibility, and check whether a GPU is available.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import numpy as np
import matplotlib.pyplot as plt

# Fix random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Use GPU if available, otherwise fall back to CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Default figure settings
plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['font.size'] = 12

---
## Section 1: Recap and Motivation

In Lab 07, we trained a neural network using **full-batch gradient descent**: the entire dataset was passed through the model at once, and the weights were updated **once per epoch**. The loop looked like this:

```python
for epoch in range(num_epochs):
    optimizer.zero_grad()
    y_pred = model(X)            # pass ALL data through the model
    loss = criterion(y_pred, y)
    loss.backward()
    optimizer.step()             # one weight update per epoch
```

This worked fine for 500 toy data points, but it **breaks down for real datasets** with tens of thousands of images for two reasons:

1. **Memory**: You cannot always fit the entire dataset in GPU/CPU memory at once.
2. **Efficiency**: Computing the gradient over the entire dataset before taking a single step is wasteful. Noisy gradient estimates from smaller batches actually help the optimizer escape bad local minima.

The solution is **mini-batch gradient descent**: split the data into small batches (e.g. 64 images at a time), and perform one gradient update per batch. Over one epoch, you will take `dataset_size / batch_size` update steps instead of just one. This is the **standard approach in all practical deep learning**.

PyTorch's `DataLoader` handles all the mechanics of batching and shuffling for you — let's see how.

---
## Section 2: Loading Data with DataLoader

### 2a. Downloading FashionMNIST

We will use **FashionMNIST** — a dataset of 70,000 grayscale images (28×28 pixels) across 10 clothing categories. It's a great benchmark: harder than the classic digit MNIST, but still small enough to train quickly on a laptop.

`torchvision` provides it as a ready-to-use dataset class. When you create the dataset object, it downloads the data automatically the first time (into a local `./data` folder). After that it loads from disk.

We pass a **transform** to preprocess the images:
- `transforms.ToTensor()` converts PIL images (integers in [0, 255]) to PyTorch tensors with values in [0.0, 1.0]. It also reorders the dimensions from `(H, W, C)` to PyTorch's `(C, H, W)` convention.

In [ ]:
# Define the sequence of transforms to apply to every image
transform = transforms.Compose([
    transforms.ToTensor(),                       # PIL → tensor, values in [0, 1]
])

# Download (first time) and load the training set
train_dataset = torchvision.datasets.FashionMNIST(
    root='./data',    # where to store the downloaded data
    train=True,       # training split (60,000 images)
    download=True,    # download if not already present
    transform=transform
)

# Download (first time) and load the validation/test set
val_dataset = torchvision.datasets.FashionMNIST(
    root='./data',
    train=False,      # test split (10,000 images)
    download=True,
    transform=transform
)

print(f"Training samples:   {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
print(f"Image shape: {train_dataset[0][0].shape}   # (channels, height, width)")
print(f"Number of classes: {len(train_dataset.classes)}")
print(f"Classes: {train_dataset.classes}")

### 2b. Visualising Some Samples

Before we train anything, it is always a good idea to look at the raw data. Let's display a small grid of FashionMNIST images.

In [ ]:
class_names = train_dataset.classes   # list of 10 class name strings

fig, axes = plt.subplots(3, 6, figsize=(12, 6))
for i, ax in enumerate(axes.flat):
    img, label = train_dataset[i]
    # img has shape (1, 28, 28); squeeze() removes the channel dim for plotting
    ax.imshow(img.squeeze(), cmap='gray')
    ax.set_title(class_names[label], fontsize=9)
    ax.axis('off')
plt.suptitle('FashionMNIST — Sample Images', fontsize=13)
plt.tight_layout()
plt.show()

### 2c. The DataLoader

A raw `Dataset` object lets you index individual samples (e.g. `train_dataset[0]` gives one `(image, label)` pair). But for training we want to iterate in **batches**. That's what `DataLoader` is for.

The two most important arguments are:
- **`batch_size`**: how many samples per batch.
- **`shuffle`**: whether to randomise the order of samples each epoch. Always shuffle training data — it prevents the model from exploiting ordering artefacts and improves generalisation. There is usually no need to shuffle the validation set.

In [ ]:
batch_size = 64

# Training loader: shuffle every epoch so batches are different each time
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

# Validation loader: no shuffling needed — just iterate in order
val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False)

print(f"Batches per training epoch: {len(train_loader)}")
print(f"Batch size: {batch_size}")
print(f"Total training samples covered per epoch: ~{len(train_loader) * batch_size}")

# Let's peek at the shape of one batch
X_batch, y_batch = next(iter(train_loader))
print(f"\nOne batch — X shape: {X_batch.shape},  y shape: {y_batch.shape}")

Notice the batch shape is `(64, 1, 28, 28)` — that is `(batch_size, channels, height, width)`. This **`(N, C, H, W)` convention** is used everywhere in PyTorch image processing. Keep it in mind: when you define convolutional layers you will need to know how many channels are coming in and going out.

---
## Section 3: The Mini-Batch Training Loop

### 3a. How Mini-Batch Differs from Full-Batch

The structural change from Lab 07 is actually very small: we add **one inner loop** that iterates over batches from the DataLoader. Instead of feeding the entire dataset at once, we process one batch at a time, and each batch produces a gradient update. Here is the comparison side by side:

In [ ]:
# -----------------------------------------------------------------------
# FULL-BATCH (Lab 07) — one update per epoch
# -----------------------------------------------------------------------
# for epoch in range(num_epochs):
#     optimizer.zero_grad()
#     y_pred = model(X)            # entire dataset at once
#     loss = criterion(y_pred, y)
#     loss.backward()
#     optimizer.step()             # one weight update per epoch

# -----------------------------------------------------------------------
# MINI-BATCH (this lab) — one update per BATCH
# -----------------------------------------------------------------------
# for epoch in range(num_epochs):
#     for X_batch, y_batch in train_loader:   # <-- inner loop over batches
#         optimizer.zero_grad()
#         y_pred = model(X_batch)             # one batch at a time
#         loss = criterion(y_pred, y_batch)
#         loss.backward()
#         optimizer.step()                    # update after every batch

With `batch_size=64` and 60,000 training images, we get **937 weight updates per epoch** instead of just 1. That is why mini-batch training converges so much faster.

### 3b. Wrapping the Loop in a `training()` Function

In practice you will reuse the training loop for many different models, so it makes sense to wrap it in a reusable function. Below is the scaffolding for the `training()` function you will need to implement in PS6. Study the signature carefully — it accepts the **model**, the **number of epochs**, the **optimizer**, the **loss function**, and the **DataLoader** as arguments. This design keeps the function general: you can swap in any model or optimizer without changing the loop itself.

In [ ]:
def training(model, n_epochs, optimizer, fn_loss, data_loader):
    """
    Train a neural network model using mini-batch gradient descent.

    Parameters
    ----------
    model       : nn.Module  — the network to train
    n_epochs    : int        — number of full passes over the training data
    optimizer   : torch.optim optimizer (e.g. Adam)
    fn_loss     : loss function (e.g. nn.CrossEntropyLoss())
    data_loader : DataLoader — yields (X_batch, y_batch) pairs

    Returns
    -------
    loss_history : list of float — average training loss per epoch
    """
    loss_history = []   # we'll collect one value per epoch

    for epoch in range(n_epochs):
        # Optional: initialize variable to accumulate loss across all batches in this epoch
        # 1-liner

        # Inner loop: iterate over every batch in the DataLoader
        for X_batch, y_batch in data_loader:

            # TO DO
            # Step 1: clear gradients from the previous batch
            # 1-liner

            # TO DO
            # Step 2: forward pass — run the batch through the model
            # 1-liner

            # TO DO
            # Step 3: compute the loss for this batch
            # 1-liner

            # TO DO
            # Step 4: backward pass — compute gradients via autograd
            # 1-liner

            # TO DO
            # Step 5: update the model weights
            # 1-liner

            # Optional: Accumulate the scalar loss (use .item() to get a plain Python float)
            # 1-liner

        # Average loss over all batches this epoch and append to history
        # 2-liner

        # Print a progress update every 5 epochs
        if (epoch + 1) % 5 == 0:
            print(f"Epoch {epoch+1:>3}/{n_epochs} | Loss: {avg_loss:.4f}")

    return loss_history

---
## Section 4: Evaluation

### 4a. Why a Separate Evaluation Step?

Training loss tells us how well the model fits the **training data** — but what we really care about is how well it **generalises** to data it has never seen before. That's what the validation set is for.

A few important things to know about running evaluation:

- During evaluation we do **not** want to update the weights — we are only doing forward passes to collect predictions.
- We do **not** need to compute gradients at all. Wrapping evaluation in `torch.no_grad()` tells PyTorch not to build the computational graph, which **saves memory and runs faster**.
- We collect predictions from all batches, concatenate them, and then compute metrics over the entire dataset at once.

Compare the structure with the training loop:

```python
# Training loop
# for epoch in range(n_epochs):
#     for X_batch, y_batch in data_loader:   # iterate over batches
#         optimizer.zero_grad()
#         y_pred = model(X_batch)            # forward pass
#         loss   = fn_loss(y_pred, y_batch)  # compute loss
#         loss.backward()                    # backprop
#         optimizer.step()                   # update weights

# Validation loop
# with torch.no_grad():                      # no graph, no gradients
#     for X_batch, y_batch in data_loader:   # same iteration pattern
#         y_pred = model(X_batch)            # forward pass only
#         preds  = y_pred.argmax(dim=1)      # predicted class per sample
#         # collect preds and labels ...
# compute metrics over all collected predictions
```

### 4b. The `validate()` Function

Here is the `validate()` function you will also implement in PS6. It runs the model over a DataLoader, collects predictions and true labels, and returns four standard classification metrics.

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def validate(model, data_loader):
    """
    Evaluate a trained model on a dataset and return performance metrics.

    Parameters
    ----------
    model       : trained nn.Module
    data_loader : DataLoader for the evaluation set

    Returns
    -------
    accuracy, precision, recall, f1 — all floats
    """
    
    # Initialize a list to hold all the predicted classes
    all_preds  = []
    
    # Intialize a list to hold ground-truth classes
    all_labels = [] 

    # IMPORTANT
    # torch.no_grad() disables gradient computation for the entire block.
    # We don't need gradients here — just forward passes to get predictions.
    with torch.no_grad():
        for X_batch, y_batch in data_loader:

            # TO DO
            # Forward pass: get raw scores for each class 
            # 1-liner

            # TO DO
            # argmax(dim=1): for each sample, pick the class with the highest score
            # 1-liner

            # TO DO
            # Append preds and labels to their respective lists
            # 2-liner

    # TO DO
    # Concatenate the list of tensors into a single tensor, then convert to numpy for sklearn
    # 2-liner

    # Compute metrics using sklearn
    # average='macro' means: compute the metric per class, then take the unweighted mean.
    # This treats all classes equally regardless of how many samples they have.
    acc  = accuracy_score(all_labels, all_preds)
    prec = precision_score(all_labels, all_preds, average='macro', zero_division=0)
    rec  = recall_score(all_labels, all_preds,    average='macro', zero_division=0)
    f1   = f1_score(all_labels, all_preds,         average='macro', zero_division=0)

    return acc, prec, rec, f1

A few things worth unpacking:

**`argmax(dim=1)`** — The model outputs a tensor of shape `(batch_size, 10)`: one score per class for each sample. `argmax(dim=1)` selects the index of the maximum score along the class dimension (dim=1), giving us the predicted class for each sample.

**`macro` averaging** — For multi-class problems there are different ways to aggregate per-class metrics. `macro` computes the metric independently for each class, then takes the unweighted average. This gives equal weight to every class, which is usually what you want when classes are roughly balanced.

**`torch.no_grad()` vs `.detach()`** — These are related but different:
- `torch.no_grad()` is a **context manager** that disables gradient tracking for an entire block of code. Use it when you know you will not need gradients at all (like during evaluation).
- `.detach()` is called on a specific **tensor** to create a copy of it with no gradient history attached. Use it when you already have a tensor that has a gradient, but you just want its raw values — for example, when capturing intermediate layer outputs for visualisation (we'll see this in Section 7).

---
## Section 5: Convolutional Layers

### 5a. The Intuition

Before we build a CNN, let's understand **why** convolutional layers exist.

A **fully connected** (linear) layer connects every input to every output. For a 28×28 grayscale image you have a total of 784 inputs. With just 256 hidden units in the first layer, you already have 784 × 256 = **200,704 parameters** — before the network has learned anything at all. This explodes quickly for larger images.

More importantly, fully connected layers are completely blind to **spatial structure**. They don't know that pixel (14, 14) is near pixel (15, 14). But images have a key property: **nearby pixels are related**. A cat's ear and its neighbouring pixels form a recognisable local pattern; it doesn't matter whether the cat is in the top-left or bottom-right of the image.

A **convolutional layer** exploits this by applying a small **filter** (also called a kernel) that slides across the image, looking for the **same local pattern everywhere**. This has two big benefits:
1. **Far fewer parameters** — a 3×3 filter has only 9 weights, shared across every position in the image.
2. **Translation invariance** — the same pattern is detected wherever it appears in the image.

### 5b. `nn.Conv2d` Parameters — One by One

Here is a table of every `nn.Conv2d` parameter you need to know:

| Parameter | What it controls |
|---|---|
| `in_channels` | How many channels does the input have? 1 for grayscale, 3 for RGB, or however many channels the previous layer output. |
| `out_channels` | How many filters to learn? Each filter produces its own output channel — called a **feature map**. More filters = more patterns detected simultaneously. |
| `kernel_size` | The size of the sliding window. `3` means a 3×3 filter. Larger kernels see more context at once but have more parameters. |
| `padding` | Rows/columns of zeros added around the image border before convolution. `padding=1` with `kernel_size=3` keeps spatial dimensions the same (H×W in → H×W out). |
| `groups` | Controls connectivity between input and output channels. `groups=1` (default): every output channel sees all input channels (standard). `groups=in_channels`: each input channel is convolved independently — this is called **depthwise convolution**, used in lightweight mobile architectures. |

Let's experiment with these one at a time.

### 5c. Manually Applying a Filter to a FashionMNIST Image

One of the best ways to understand convolution is to manually assign a known filter and see what it does to an image. Let's define a **hollowness detection filter** and apply it to one FashionMNIST image.

The filter we'll use:
```
[[ 1,  1,  1],
 [ 1, -8,  1],
 [ 1,  1,  1]]
```
This responds strongly where the difference in intensity between a pixel and its inmediate neighbors is large.

In [ ]:
# Grab one image from the training set
img, label = train_dataset[22]
print(f"Image shape: {img.shape}   # (channels=1, height=28, width=28)")
print(f"Label: {class_names[label]}")

# Define a conv layer: 1 input channel, 1 output channel, 3x3 kernel, padding=1
# padding=1 ensures the output stays 28x28 (same spatial size as the input)
conv = nn.Conv2d(in_channels=1, out_channels=1, kernel_size=3, padding=1)

# Manually assign a local hollowness dete tion filter that looks for dark pixels surrounded by bright contours
# Shape must be (out_channels, in_channels, kernel_H, kernel_W) = (1, 1, 3, 3)
edge_filter = torch.tensor([[
    [[ 1., 1., 1.],
     [ 1., -8., 1.],
     [ 1., 1., 1.]]
]])
conv.weight.data = edge_filter
conv.bias.data.zero_()   # no bias — keep it clean

# Apply the filter.
# img has shape (1, 28, 28); we need (batch, channels, H, W) = (1, 1, 28, 28)
# unsqueeze(0) adds the batch dimension.
with torch.no_grad():
    output = conv(img.unsqueeze(0))   # shape: (1, 1, 28, 28)

# Visualise the original image and the filtered result
fig, axes = plt.subplots(1, 2, figsize=(8, 4))

axes[0].imshow(img.squeeze(), cmap='gray')
axes[0].set_title(f'Original: {class_names[label]}')
axes[0].axis('off')

axes[1].imshow(output.squeeze().numpy(), cmap='gray')
axes[1].set_title('After local cavity filter')
axes[1].axis('off')

plt.tight_layout()
plt.show()

### 5d. Multiple Filters → Multiple Feature Maps

In practice we learn many filters simultaneously. Each filter produces its own output — called a **feature map**. The `out_channels` parameter controls how many filters (and therefore how many feature maps) a conv layer learns.

In [ ]:
# 1 input channel (grayscale), 8 output channels (8 different filters to learn)
conv_multi = nn.Conv2d(in_channels=1, out_channels=8, kernel_size=3, padding=1)

with torch.no_grad():
    feature_maps = conv_multi(img.unsqueeze(0))   # shape: (1, 8, 28, 28)

print(f"Output shape: {feature_maps.shape}")
print("  → (batch=1, out_channels=8, H=28, W=28)")

# Plot each of the 8 feature maps
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for i, ax in enumerate(axes.flat):
    ax.imshow(feature_maps[0, i].numpy(), cmap='viridis')
    ax.set_title(f'Filter {i}')
    ax.axis('off')
plt.suptitle('8 feature maps from randomly initialised filters', fontsize=13)
plt.tight_layout()
plt.show()

Each panel is one feature map — the response of one filter sliding across the entire image. Since the filters here are **randomly initialised**, the patterns are not yet meaningful. After training, each filter learns to respond to something useful for the classification task — edges, textures, corners, and eventually more abstract concepts in deeper layers.

### 5e. Spatial Dimensions After Convolution

It's important to be able to compute the output size of a conv layer, especially when you need to figure out how many units go into the first fully connected layer. The formula is:

$$H_{\text{out}} = H_{\text{in}} - \text{kernel\_size} + 2 \times \text{padding} + 1$$

So with `kernel_size=3, padding=1`: $H_{\text{out}} = H_{\text{in}} - 3 + 2 + 1 = H_{\text{in}}$ — the spatial size is preserved.

Without padding: $H_{\text{out}} = H_{\text{in}} - 2$ — you lose one pixel on each side.

In [ ]:
# Verify the formula with a dummy input
x = torch.randn(1, 1, 28, 28)   # batch=1, channels=1, H=28, W=28

with torch.no_grad():
    print("No padding (padding=0): ", nn.Conv2d(1, 1, 3, padding=0)(x).shape)   # → (1, 1, 26, 26)
    print("With padding=1:         ", nn.Conv2d(1, 1, 3, padding=1)(x).shape)   # → (1, 1, 28, 28)

### 5f. MaxPool2d — Downsampling the Feature Maps

After a conv layer it is common to apply **max pooling**. A max pool layer slides a small window (usually 2×2) over the feature map and keeps only the **maximum value** in each window. With `stride=2`, the window moves two steps at a time, so the output is half the size of the input in each spatial dimension.

Max pooling has two purposes:
1. **Reduces computation** — smaller feature maps mean fewer operations in the next layer.
2. **Translation invariance** — if a pattern shifts by a pixel or two, the max value in the window is likely the same, making the representation more robust.

In [ ]:
pool = nn.MaxPool2d(kernel_size=2, stride=2)   # halves H and W

x = torch.randn(1, 8, 28, 28)   # batch=1, 8 channels, 28x28
print(f"Before pooling: {x.shape}")
print(f"After pooling:  {pool(x).shape}")   # → (1, 8, 14, 14)

Spatial dimensions are halved (28 → 14), but the number of channels is unchanged. This is exactly what happens in the CNN we will build next.

---
## Section 6: Building and Training a CNN on FashionMNIST

### 6a. Defining the CNN

A typical CNN follows a repeating pattern:

```
[Conv → ReLU → MaxPool]  ×  (number of blocks)
         ↓
      Flatten
         ↓
  Fully Connected layers
         ↓
      Output
```

Each conv+pool block extracts increasingly abstract features while shrinking the spatial dimensions. After the last block, we flatten the 3D feature tensor into a 1D vector and pass it through standard linear layers for the final classification.

Here is a simple two-block CNN for FashionMNIST. Pay attention to how the spatial dimensions change at each step — we trace it in the comments.

In [ ]:
class FashionCNN(nn.Module):
    def __init__(self):
        super().__init__()

        # --- Block 1 ---
        # Input:  (N, 1, 28, 28)   — 1 grayscale channel
        # After conv1 (padding=1): (N, 16, 28, 28)  — 16 feature maps, same spatial size
        # After pool:              (N, 16, 14, 14)  — halved
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, padding=1)

        # --- Block 2 ---
        # Input:  (N, 16, 14, 14)
        # After conv2 (padding=1): (N, 32, 14, 14)  — 32 feature maps, same spatial size
        # After pool:              (N, 32, 7, 7)    — halved again
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)

        # Shared pooling layer (same params used in both blocks)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.relu = nn.ReLU()

        # --- Fully connected layers ---
        # After the two conv+pool blocks, the feature tensor is (N, 32, 7, 7)
        # We flatten it to (N, 32*7*7) = (N, 1568) before feeding to linear layers
        self.fc1 = nn.Linear(32 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)             # 10 output classes

    def forward(self, x):
        # TO DO
        # Block 1: conv → relu → pool
        # (N, 1, 28, 28) → (N, 16, 14, 14)
        # 1-liner 

        # TO DO
        # Block 2: conv → relu → pool
        # (N, 16, 14, 14) → (N, 32, 7, 7)
        # 1-liner

        # TO DO
        # Flatten: reshape (N, 32, 7, 7) → (N, 1568)
        # x.size(0) keeps the batch dimension; -1 tells PyTorch to infer the rest
        x = x.view(x.size(0), -1)

        # TO DO
        # Fully connected layers
        x = self.relu(self.fc1(x))   # (N, 1568) → (N, 128)
        x = self.fc2(x)              # (N, 128) → (N, 10)  — raw logits, no activation
        return x


# Instantiate and move to device
model = FashionCNN().to(device)
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

Two things to notice:

**`x.view(x.size(0), -1)`** — This is the "flatten" step. After the conv+pool blocks, `x` has shape `(N, 32, 7, 7)`. Fully connected layers expect a 1D input per sample, so we reshape to `(N, 32*7*7)`. `x.size(0)` preserves the batch dimension, and `-1` tells PyTorch to infer the remaining dimension automatically.

**No activation on the last layer** — `self.fc2` outputs raw scores called **logits**. We do not apply softmax here because `nn.CrossEntropyLoss` in PyTorch expects raw logits and applies the log-softmax internally (which is numerically more stable).

### 6b. Training

We use our `training()` function from Section 3b. We'll train for 10 epochs — enough to see meaningful progress on FashionMNIST without taking too long.

In [ ]:
# Loss function: CrossEntropyLoss is standard for multi-class classification
criterion = nn.CrossEntropyLoss()

# Optimizer: Adam with a learning rate of 0.001
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# Train! This may take a few minutes on CPU.
print("Starting training...")
loss_history = training(model, n_epochs=10, optimizer=optimizer,
                        fn_loss=criterion, data_loader=train_loader)

# Plot the training loss curve
plt.figure(figsize=(8, 4))
plt.plot(range(1, len(loss_history) + 1), loss_history, color='steelblue', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Average Loss')
plt.title('Training Loss over Epochs')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

The loss should decrease steadily over epochs. If it plateaus early, you could try training longer, using a lower learning rate, or adding more capacity to the model.

### 6c. Evaluating on Training and Validation Sets

Now let's call `validate()` on both sets to see how well the model generalises. A big gap between training and validation metrics suggests **overfitting** — the model has memorised training data but struggles on new examples.

In [ ]:
# Evaluate on both splits
train_acc, train_prec, train_rec, train_f1 = validate(model, train_loader)
val_acc,   val_prec,   val_rec,   val_f1   = validate(model, val_loader)

# Print a summary table
print(f"{'':10s}  {'Accuracy':>9s}  {'Precision':>9s}  {'Recall':>9s}  {'F1':>9s}")
print("-" * 55)
print(f"{'Train':10s}  {train_acc:>9.3f}  {train_prec:>9.3f}  {train_rec:>9.3f}  {train_f1:>9.3f}")
print(f"{'Val':10s}  {val_acc:>9.3f}  {val_prec:>9.3f}  {val_rec:>9.3f}  {val_f1:>9.3f}")